# SBERT using k-NN for Retrieval

### Representation Learning

First SBERT, then for SOTA use SBERT + BERT re-ranking

In [3]:
# SBERT k-NN 
import torch
from sentence_transformers import SentenceTransformer
from transformers import BertTokenizer, BertModel
from scipy.spatial.distance import cosine
import json
import re
import psycopg2
import psycopg2.extras
import numpy as np
from collections import Counter
import wordninja

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)

2.5.1+cu121


In [4]:
# Extract Artist and Painting Metadata from ART_RECSYS_DB to be used for TF-IDF VSM, BERT, and CLIP

# Database connection
def get_db_connection():
    conn = psycopg2.connect(
        host="localhost",
        database="ART_RECSYS_DB",
        user="postgres",
        password="Catmelon304!"
    )
    return conn

get_db_connection()

<connection object at 0x0000023626FBA350; dsn: 'user=postgres password=xxx dbname=ART_RECSYS_DB host=localhost', closed: 0>

## Preprocessing 

### Validation and Cleaning

Validation has already been performed in the beginning of the notebook, hence it's possible to move directly to the NULL handling section of this pipeline.

### Handle NULLs/missing values and Text Normalisation

As discussed earlier, the following fields contain missing/null values (see histogram plot of missing values per field) and for the recommendation architecture I will be using a larger set of text field since BERT is a transformer-based architecture, which means that model adopts the mechanism of attention, weighing the influence of different parts of the input data. 

Fields that contain missing values; 

**Painting Details** title, year_created, genre, media, description_tags

**Artist Details** nationality, feilds, art_movements, bio

Using explicit NULL tokens e.g. [GENRE] unknown genre, will allow the model to learn the lack of certain fields within a painting representation.

*N.B: The only fields that are not handled for missing values are artist and art_style because they have no missing/null values.* 

- handle missing values
- denote fields
- clean noisy data
- deduplicate tokens
- normalise structured metadata
- convert numbers to semantic representation

Raw text can't be used to extract embeddings, since the data is heterogeneous and raw concatenation has no field boundaries, and is partially structured containing missing data. 

#### Free Text 
**Title, Bio**

In [ ]:
# In the title column standardise all noisy 'unknown' variants that add no semantic value to NULL so that 
# BERT learns one representation of missing title instead of 'untitled', 'untitle', 'untitled (1)', No Title, Not_Detected_22056 etc...
def set_unknown_title():
    conn = get_db_connection()
    cur = conn.cursor()

    query = """ 
        UPDATE paintings_and_artists_metadata_bert
        SET title = NULL
        WHERE title IS NOT NULL
        AND (
            LOWER(title) ~ '^\s*(untitled|unknown|untitle)\s*([0-9]+|\([0-9]+\))?\s*$'
            OR LOWER(title) ~ '^\s*unknown\s+title\s*([0-9]+|\([0-9]+\))?\s*$'
            OR LOWER(title) ~ '^\s*not\s+detected\s*([0-9]+|\([0-9]+\))?\s*$'
            OR LOWER(title) ~ '^\s*not[_\s]*detected[_\s]*\d*$'
            OR LOWER(title) ~ '^\s*no\s+title\s*(\(?\d+\)?\s*)*$'
            ); 
        """
    
    cur.execute(query)

    conn.commit()
    cur.close()

def remove_bracket_nums():
    conn = get_db_connection()
    cur = conn.cursor()

    query = """ 
            UPDATE paintings_and_artists_metadata_bert
            SET title = REGEXP_REPLACE(title, '\s*\(\s*\d+\s*\)', '', 'g')
            WHERE title IS NOT NULL;
            """
    
    cur.execute(query)

    conn.commit()
    cur.close()

set_unknown_title()    
remove_bracket_nums()

In [ ]:
# Title preprocessing i.e. NULL/missing handling and text normalisation 
ROMAN_NUMERAL_PATTERN = re.compile(r'\b(i{1,3}|iv|v|vi{0,3}|ix|x)\b', re.IGNORECASE)

def normalize_roman_numerals(text: str) -> str:
    return ROMAN_NUMERAL_PATTERN.sub(lambda m: m.group(0).upper(), text)

def process_title(title: str) -> str:
    # Handle NULL/missing values 
    if title is None or title.strip() == "":
        return "unknown title"

    # Normalise whitespace
    title = title.strip() 
    title = re.sub(r'\s+', ' ', title)

    # Fix broken apostrophes e.g. "Martin S" → "Martin's" and normalise roman numerals
    title = re.sub(r"\b([A-Za-z]+)\s+S\b", r"\1's", title)
    title = normalize_roman_numerals(title)

    # Remove trailing index numbers only when safe
    if re.search(r'(untitled|drawing|study|composition|abstraction)', title, re.IGNORECASE):
        title = re.sub(r'\s*\(?\d+\)?$', '', title)

    title = re.sub(r'\s+', ' ', title).strip()

    return title

# Used to avoid cutting mid-word
def safe_cut(text, length):
    cut = text[:length]
    return cut[:cut.rfind(" ")] if " " in cut else cut

# Bio preprocessing, including character cutoff since bio fields contain the most 
# characters and heavily bias the embedding space if left untrimmed
def process_bio(bio: str, max_chars=2000) -> str:
    if bio is None or bio.strip() == "":
        return "no biography available"
    
    # Normalise whitespace, including newlines and tabs
    bio = bio.strip()
    bio = re.sub(r'\s+', ' ', bio)

    # Truncate so that bio is token-safe for SBERT 
    if len(bio) > max_chars:
        head_len = int(max_chars * 0.6)
        tail_len = max_chars - head_len

        head = safe_cut(bio, head_len)
        tail = safe_cut(bio[::-1], tail_len)[::-1]

        bio = f"{head} ... {tail}"

    return bio


#### Categorical 
**Artist, Genre, Art Style, Nationality**


In [ ]:
# This method can be used for all categorical fields, to handle NULL/missing values, convert all fields to lowercase apart 
# from artist and remove excess whitespace 
def process_categorical_field(field_value: str, field_name: str = None) -> str:
    # Handle NULL/missing values 
    if field_value is None or field_value.strip() == "":
        return f"unknown {field_name}"

    field_value = field_value.strip()

    # Normalise whitespace and convert all categoricals to lowercase except artist since it harms entity recognition
    field_value = re.sub(r'\s+', ' ', field_value)
    if field_name != "artist":
        field_value = field_value.lower()

    return field_value

#### Multi-Value Fields
**Description Tags, Media, Art Movements, Fields**


In [ ]:
# This method can be used for all multi-value fields, it handles nulls/missing values, 
# converts all comma seperated items to lower case, and applies the respective preprocessing for description_tags, 
# joining all value segments using |
def process_multi_value_field(feild_value: str, field_name: str = None) -> str:
    # Handle NULL/missing values
    if feild_value is None or feild_value.strip() == "":
        return f"unknown {field_name}" 

    # Split on comma 
    items = feild_value.split(",")
    cleaned = []
    for item in items:
        item = item.strip().lower()
        if item == "":
            continue
        
        # Replace hyphens with space when in-between words only and normalise internal whitespace
        if field_name == "description tags":
            item = re.sub(r'(?<=\w)-(?=\w)', ' ', item)

            # Apply segmentation only if; no spaces already, long enough, purely alphabetic
            if " " not in item and len(item) > 10 and item.isalpha():
                split_words = wordninja.split(item)

                # Safety checks to avoid bad splits (e.g. single char fragments)
                if (len(split_words) > 1 and all(len(w) > 2 for w in split_words) and len(" ".join(split_words)) >= len(item) * 0.8):
                    item = " ".join(split_words) 

        item = re.sub(r'\s+', ' ', item)
        cleaned.append(item)

    if not cleaned:
        return f"unknown {field_name}" 

    # Deduplicate while preserving order
    unique_items = list(dict.fromkeys(cleaned))

    return " | ".join(unique_items)

#### Numerical Features
**Year Created**

In [ ]:
# Instead of using years, encode them as semantic 
def process_year(year_created):
    # NULL/missing 
    if year_created is None or str(year_created).strip() == "":
        return "unknown period"

    try:
        year = int(year_created)
    except (ValueError, TypeError):
        return "unknown period"

    # Period mapping 
    if year < 1400:
        return "medieval period 14th century"
    elif year < 1600:
        return "renaissance period 16th century"
    elif year < 1700:
        return "baroque period 17th century"
    elif year < 1800:
        return "rococo enlightenment period 18th century"
    elif year <= 1850:
        return "early modern period 18th century"
    elif year <= 1900:
        return "late 19th century impressionism era"
    elif year <= 1945:
        return "early 20th century modernism"
    elif year <= 1970:
        return "mid 20th century post war modern"
    elif year <= 2000:
        return "late 20th century contemporary"
    else:
        return "contemporary period 20th century" 

In [ ]:
def format_text_fields(title, year_created, genre, art_style, media, description_tags, artist, nationality, fields, art_movements, bio):
    title_processed = process_title(title)
    year_processed = process_year(year_created)
    genre_processed = process_categorical_field(genre, "genre")
    art_style_processed = process_categorical_field(art_style, "art style")
    media_processed = process_multi_value_field(media, "media")
    description_tags_processed = process_multi_value_field(description_tags, "description tags")
    artist_processed = process_categorical_field(artist, "artist")
    nationality_processed = process_categorical_field(nationality, "nationality")
    fields_processed = process_multi_value_field(fields, "fields")
    art_movements_processed = process_multi_value_field(art_movements, "art movements")
    bio_processed = process_bio(bio)

    # Structured reperesntation 
    structured = {
        "title": title_processed,
        "year_period": year_processed,
        "genre": genre_processed,
        "art_style": art_style_processed,
        "media": media_processed.split(" | "),
        "description_tags": description_tags_processed.split(" | "),
        "artist": artist_processed,
        "nationality": nationality_processed,
        "fields": fields_processed.split(" | "),
        "art_movements": art_movements_processed.split(" | "),
        "bio": bio_processed
    }

    # Flat text for SBERT input 
    text = (
        f"[TITLE] {title_processed}. "
        f"[GENRE] {genre_processed}. "
        f"[ART_STYLE] {art_style_processed}. "
        f"[DESCRIPTION_TAGS] {description_tags_processed}. "
        f"[MEDIA] {media_processed}. "
        f"[YEAR_PERIOD] {year_processed}. "
        f"[ARTIST] {artist_processed}. "
        f"[NATIONALITY] {nationality_processed}. "
        f"[FIELDS] {fields_processed}. "
        f"[ART_MOVEMENTS] {art_movements_processed}. "
        f"[BIO] {bio_processed}"
    )

    return text, structured

def build_processed_texts():
    conn = get_db_connection()
    cur = conn.cursor()

    query = """
    SELECT painting_id, title, year_created, genre, art_style, media, 
        description_tags, artist, nationality, fields, art_movements, bio
    FROM paintings_and_artists_metadata_bert
    ORDER BY painting_id
    """
    cur.execute(query)
    rows = cur.fetchall()

    processed_data = []
    for row in rows:
        (painting_id, title, year_created, genre, art_style, media, description_tags, 
         artist, nationality, fields, art_movements, bio) = row
        text, structured = format_text_fields(title, year_created, genre, art_style, media, description_tags, 
                                              artist, nationality, fields, art_movements, bio)
        processed_data.append((text, json.dumps(structured), painting_id))
        print(processed_data[-1])

    conn.commit()
    cur.close()
    conn.close()

    return processed_data

In [ ]:
processed_data = build_processed_texts() 

In [ ]:
def store_processed_data(processed_data):
    conn = get_db_connection()
    cur = conn.cursor() 

    for item in processed_data:
        text, structured_json, painting_id = item

        cur.execute("""
            UPDATE paintings_and_artists_metadata_bert
            SET processed_text = %s,
                processed_fields = %s
            WHERE painting_id = %s
        """, (
            text,
            structured_json,
            painting_id
        ))

    conn.commit()
    cur.close()
    conn.close()

store_processed_data(processed_data)

### Extract Embeddings using SBERT

In [5]:
# Structured concatenation using tokens e.g. [GENRE], [STYLE], [TITLE], etc...
model = SentenceTransformer('all-mpnet-base-v2') # all-roberta-large-v1, paraphrase-multilingual-mpnet-base-v2
#embeddings = model.encode([item["text"] for item in processed_data])